# GadgetHub RAG — Guardrailed Product Advisor

Recommends smartwatches, smart TVs, laptops, and cameras from `../data/` based on what the user wants (**demand**), a problem they describe (**problem**), or an indirect signal about their needs (**hint**). Every fact in an answer is retrieved from the vector store in the same turn — nothing is answered from the model's memory, and every reply stays short.

**LLM:** `gpt-4o-mini` (OpenAI) · **Embedding:** `text-embedding-3-small` (OpenAI) · **Vector Store:** Qdrant Cloud


## 1. Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from ragwire import RAGWire, setup_logging
import ragwire

logger = setup_logging(log_level="INFO")
print(ragwire.__version__)


In [ ]:
rag = RAGWire('rag_pipeline.yaml')


## 2. Prepare the catalog — one product per chunk

Each catalog PDF in `../data/` bundles many products into a single file (30 smartwatches, 35 cameras, dozens of laptops and TVs). RAGWire's metadata extractor runs once per **file** and stamps the result onto every chunk of that file — so ingesting a catalog PDF directly would tag every product's chunks with only the first product's metadata.

To keep `rag_metadata.yaml` accurate (one product per chunk, as it requires), we split each catalog into one small file per product before ingesting. Each catalog uses a different layout, so each gets its own splitter built around a reliable anchor in that catalog's text:

- **Smartwatch** — anchored on the `MODEL #NN // BRAND: ...` heading, truncated at the `...PIPELINES` telemetry footer.
- **Camera** — anchored on the `Sensor / RAM` spec label; brand/name/price are the three lines immediately above it.
- **Laptop / Smart TV** — anchored on the `Brand / Make` spec label; name/price are the two lines immediately above it, truncated at the `...Checkout.` sentence that closes every listing.

Boundaries are found from fixed offsets in the text (not string search on may-repeat values), so a block never bleeds into the next product even when two products share a page.

In [ ]:
import re
from pathlib import Path

CATALOGS = {
    "smartwatch": "../data/rag_smartwatch_catalog_30_models.pdf",
    "camera": "../data/rag_camera_catalog_35_models.pdf",
    "laptop": "../data/rag_electronic_shop_laptop_catalog_global_usd.pdf",
    "smart_tv": "../data/rag_electronic_shop_smart_tv_catalog_35_pages.pdf",
}

STAGE_DIR = Path("../data/_staged_products")
STAGE_DIR.mkdir(exist_ok=True)


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:60]


def split_smartwatch(text: str) -> list[tuple[str, str]]:
    # Anchor: 'MODEL #NN // BRAND: ...' heading. End: the telemetry footer.
    # The product name is the first non-empty line after the heading.
    model_re = re.compile(r"M\s*O\s*D\s*E\s*L\s*#\s*\d+.*?B\s*R\s*A\s*N\s*D\s*:\s*[A-Z0-9 ]+")
    positions = [m.start() for m in model_re.finditer(text)]
    entries = []
    for i, pos in enumerate(positions):
        end = positions[i + 1] if i + 1 < len(positions) else len(text)
        block = text[pos:end]
        end_marker = re.search(r"PIPELINES\.?", block)
        if end_marker:
            block = block[: end_marker.end()]
        block = block.strip()
        name = next(l for l in block.splitlines()[1:] if l.strip())
        entries.append((name.strip(), block))
    return entries


def split_camera(text: str) -> list[tuple[str, str]]:
    # Anchor: the 'Sensor / RAM' spec label. Brand/name/price are the three
    # non-empty lines immediately above it. Block ends right before the NEXT
    # product's brand line (found the same way), so nothing leaks across.
    anchor_re = re.compile(r"Sensor\s*/\s*RAM")
    lines = text.splitlines(keepends=True)
    offsets, pos = [], 0
    for line in lines:
        offsets.append(pos)
        pos += len(line)

    entries = []
    for li, line in enumerate(lines):
        if not anchor_re.search(line):
            continue
        nonempty = []
        j = li - 1
        while j >= 0 and len(nonempty) < 3:
            if lines[j].strip():
                nonempty.append(j)
            j -= 1
        nonempty.reverse()
        brand_idx, name_idx, price_idx = nonempty
        entries.append((
            lines[brand_idx].strip(), lines[name_idx].strip(), lines[price_idx].strip(),
            offsets[brand_idx], offsets[li],
        ))

    results = []
    for i, (brand, name, price, header_start, spec_start) in enumerate(entries):
        end = entries[i + 1][3] if i + 1 < len(entries) else len(text)
        block = text[spec_start:end]
        block = re.sub(r"Quantum Optics.*?Pro Catalog 2026", "", block)
        block = re.sub(r"Page \d+ of \d+", "", block)
        full = f"{brand}\n{name}\n{price}\n{block}".strip()
        results.append((f"{brand} {name}", full))
    return results


def split_brand_make(text: str) -> list[tuple[str, str]]:
    # Anchor: the 'Brand / Make' spec label (laptop & smart TV catalogs share
    # this layout). Name/price are the two non-empty lines immediately above it.
    # Each block is truncated at its own closing '...Checkout.' sentence, which
    # discards the next product's name/price lines that would otherwise leak in.
    anchor_re = re.compile(r"Brand\s*/\s*Make")
    anchors = [m.start() for m in anchor_re.finditer(text)]
    results = []
    for i, pos in enumerate(anchors):
        preceding = [l for l in text[:pos].splitlines() if l.strip()]
        name_line, price_line = preceding[-2], preceding[-1]
        end = anchors[i + 1] if i + 1 < len(anchors) else len(text)
        block = text[pos:end]
        end_marker = re.search(r"Checkout\.", block)
        if end_marker:
            block = block[: end_marker.end()]
        full = f"{name_line}\n{price_line}\n{block}".strip()
        results.append((name_line.strip(), full))
    return results


SPLITTERS = {
    "smartwatch": split_smartwatch,
    "camera": split_camera,
    "laptop": split_brand_make,
    "smart_tv": split_brand_make,
}

staged_paths = []
for category, path in CATALOGS.items():
    result = rag.loader.load(path)
    if not result["success"]:
        raise RuntimeError(f"Failed to load {path}: {result['error']}")

    entries = SPLITTERS[category](result["text_content"])

    for i, (name, block) in enumerate(entries, start=1):
        slug = slugify(re.sub(r"\s+", " ", name))
        out_path = STAGE_DIR / f"{category}_{i:02d}_{slug}.md"
        out_path.write_text(block, encoding="utf-8")
        staged_paths.append(out_path)

    print(f"{category}: {len(entries)} products staged")

print(f"\nStaged {len(staged_paths)} single-product files in {STAGE_DIR}/")


## 3. Ingest

File-hash dedup means re-running this cell after the first successful ingest is cheap — already-ingested products are skipped. `force_recreate: true` in `rag_pipeline.yaml` means the very first init of `RAGWire` above already dropped and recreated the collection, so a full re-run of this notebook always ingests a clean, current catalog.

In [ ]:
stats = rag.ingest_directory(str(STAGE_DIR))
stats


## 4. Sanity-check retrieval

In [ ]:
def show(results):
    if not results:
        print("No results.")
        return
    for doc in results:
        m = doc.metadata
        print(f"- [{m.get('category')}] {m.get('brand_or_make')} {m.get('product_name')} — {m.get('price')}")


In [ ]:
# Plain semantic search — no category filter
show(rag.retrieve("something for tracking my heart rate while I run", top_k=3))


In [ ]:
# Category-filtered search
show(rag.retrieve("a laptop for video editing", top_k=3, filters={"category": "laptop"}))
show(rag.retrieve("a big screen for movie nights", top_k=3, filters={"category": "smart_tv"}))


## 5. Guardrailed recommendation agent

Two tools:
- `get_filter_context` — lets the agent confirm the exact stored `category` value before searching (camera / laptop / smart_tv / smartwatch).
- `search_products` — the only source of product facts. Returns a compact, field-by-field block per product (not a raw dump) so the model has less to read and less room to misquote.

In [ ]:
from typing import Optional, Dict

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver


# =====================================================
# MODEL
# =====================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# =====================================================
# TOOLS
# =====================================================

@tool
def get_filter_context(query: str) -> str:
    """
    Inspect available metadata fields and their stored values before retrieval.

    Use this to confirm the exact spelling of `category` before calling
    search_products. Available fields:
    - category (camera | laptop | smart_tv | smartwatch)
    - brand_or_make
    - product_name

    You may call this more than once in the same turn if you need to check
    more than one candidate category.
    """
    return rag.get_filter_context(query)


def _normalize_filters(filters: Optional[Dict]) -> Optional[Dict]:
    """Defensive normalization — filter values must be lowercase, matching
    how `category` is stored per rag_metadata.yaml."""
    if not filters:
        return filters
    return {
        k: (v.strip().lower() if isinstance(v, str) else v)
        for k, v in filters.items()
    }


# Category-specific fields — only the ones matching a product's category
# will actually be set; the rest come back null and are skipped here.
CATEGORY_FIELDS = {
    "camera": ["sensor_ram", "storage_cards", "graphics_engine", "key_specs", "camera_description"],
    "laptop": ["memory_ram", "storage_ssd", "graphics_card", "core_architecture"],
    "smart_tv": ["screen_size", "resolution_standard", "display_tech_specialty"],
    "smartwatch": ["memory_and_storage", "graphics_coprocessor", "sensor_array", "architectural_description"],
}


def _format_result(doc) -> str:
    m = doc.metadata
    category = m.get("category", "unknown")
    lines = [
        f"Product: {m.get('product_name', 'unknown')}",
        f"Brand: {m.get('brand_or_make', 'unknown')}",
        f"Category: {category}",
        f"Price: {m.get('price') or 'null'}",
    ]
    for field in CATEGORY_FIELDS.get(category, []):
        value = m.get(field)
        if value:
            lines.append(f"{field.replace('_', ' ').title()}: {value}")
    return "\n".join(lines)


@tool
def search_products(query: str, filters: Optional[Dict] = None) -> str:
    """
    Search the GadgetHub product catalog.

    IMPORTANT RULES:
    - Always retrieve from the vector database. Never fabricate a product.
    - If a category is determined (camera, laptop, smart_tv, smartwatch),
      pass it as a filter.
    - Category filter values MUST be lowercase exactly as stored.

    Example:
        search_products(
            query="rugged watch for diving",
            filters={"category": "smartwatch"},
        )
    """
    filters = _normalize_filters(filters)

    results = rag.retrieve(query=query, filters=filters)

    # Fallback: if a filtered search returns nothing, the category value may
    # not exactly match what's stored. Retry once unfiltered so a real match
    # isn't lost to an exact-match miss.
    #
    # IMPORTANT: pass filters={} here, NOT filters=None — ragwire's
    # retrieve() treats `filters is None` as "caller supplied nothing" and
    # will trigger its own auto_filter LLM extraction, re-deriving the same
    # category and silently repeating the failed filtered search. An empty
    # dict is not None (skips auto-extraction) and is falsy (skips filter
    # application), so it's the only way to force a genuinely unfiltered retry.
    if not results and filters:
        results = rag.retrieve(query=query, filters={})

    if not results:
        return "NO_RESULTS_FOUND"

    return "\n\n".join(_format_result(doc) for doc in results)


tools = [get_filter_context, search_products]


In [ ]:
# =====================================================
# SYSTEM PROMPT
# =====================================================

SYSTEM_PROMPT = """
You are the GadgetHub Store Assistant with access to
gadgethub-catalog-openai-qdrant.

The collection contains gadget product records with:
- category (camera | laptop | smart_tv | smartwatch)
- brand_or_make
- product_name
- price
- category-specific fields (only the ones matching a product's category are set):
  - camera: sensor_ram, storage_cards, graphics_engine, key_specs, camera_description
  - laptop: memory_ram, storage_ssd, graphics_card, core_architecture
  - smart_tv: screen_size, resolution_standard, display_tech_specialty
  - smartwatch: memory_and_storage, graphics_coprocessor, sensor_array, architectural_description


==================================================
CLASSIFY EVERY USER MESSAGE
==================================================

A) DEMAND
The user explicitly names a product or category they want
(e.g. "show me a smartwatch", "I need a gaming laptop", "recommend a camera",
"I want a new TV").

B) PROBLEM
The user describes an issue a product can help with
(e.g. "my laptop can't handle video editing anymore", "I can't see the
screen from across the room", "I want to track my heart rate on runs",
"my photos never come out sharp at night").

C) HINT
An indirect signal about their needs that implies a category
(e.g. "I'm training for a marathon", "I edit 4K footage every day",
"I host movie nights every week", "I post travel photography daily").
If the hint is too vague to map to one category with reasonable confidence,
ask ONE short clarifying question instead of guessing a category to force a
search.

D) GENERAL CHAT
Greetings or small talk unrelated to gadgets. Reply in one short sentence.
Do not search.

If the message is unrelated to gadgets (general trivia, unrelated tasks,
requests for advice outside electronics), decline in one sentence and
redirect:
"I can only help with product recommendations for smartwatches, smart TVs,
laptops, and cameras."


==================================================
GENERAL RULES — NO HALLUCINATION
==================================================

- Never answer a product question from memory.
- Every product fact in your reply must come from search_products
  in the current turn.
- Never invent a product, price, spec, or feature.
- Never merge details from two different products into one line.
- Copy retrieved values exactly — do not rewrite prices or specs.


==================================================
BEFORE SEARCHING
==================================================

1. Classify the message (A-D above).
2. For A/B/C, decide the single most likely category: camera, laptop,
   smart_tv, or smartwatch. If the need genuinely spans more than one
   category, search without a category filter rather than guessing one.
3. Call get_filter_context to confirm the exact stored category value before
   searching. You may call it more than once if you're unsure between two
   candidates.
4. Call search_products with:

   filters={"category": "smartwatch"}

   (or the matching category, or filters={} when no single category applies).


==================================================
NO RESULTS
==================================================

If search_products returns NO_RESULTS_FOUND, reply with exactly one short
sentence:
"I couldn't find a matching product in our catalog right now."
Do not also restate this a second way in the same reply.


==================================================
OUTPUT RULES — KEEP IT SHORT AND FAST
==================================================

- No greetings, no filler, no repeating the user's question back.
- Recommend at most 3 products unless the user explicitly asks for more.
- One line per product, values copied exactly from search_products:

  <Brand> <Product Name> — <Price> — <key spec> — <one-clause why>

- Add a closing sentence only if the user asked a follow-up like
  availability or compatibility, and keep it to one short sentence — never
  a paragraph, never repeated boilerplate.
"""


# =====================================================
# MEMORY + AGENT
# =====================================================

checkpointer = InMemorySaver()

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

agent


## 6. Interactive Q&A

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "demo"}}

print("\nGadgetHub Assistant ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue
    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config=config,
    )
    print(f"\nAgent: {response['messages'][-1].text}\n")
